In [0]:
pip install office365-rest-python-client --trusted-host pypi.org --trusted-host files.pythonhosted.org openpyxl

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 7.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
import io
import pandas as pd
from office365.runtime.auth.client_credential import ClientCredential
from office365.sharepoint.client_context import ClientContext
from office365.sharepoint.files.file import File

try:
    tenant_id = ""
    client_id = ""
    client_secret = ""

    site_url = "https://share.philips.com/sites/RI-CPMExcellence" 
    file_relative_url = "/sites/RI-CPMExcellence/Shared Documents/General/Dashboard/input/kpi_thresholds.xlsx"

    credentials = ClientCredential(client_id, client_secret)
    ctx = ClientContext(site_url).with_credentials(credentials)

    # Download the file's raw bytes
    response = File.open_binary(ctx, file_relative_url)
    file_content = response.content

    # Load into a DataFrame
    df = pd.read_excel(io.BytesIO(file_content),sheet_name="Thresholds")

    print(df.head())


except Exception as e:
    print(f"Connection establishment with Sharepoint failed with error: {e}")

   High-risk projects  SOW  ...  NPS  Extended Hypercare Cases
0                 NaN  100  ...   66                        42

[1 rows x 22 columns]
{'High-risk projects': nan, 'SOW': 100, 'High value projects (DASB)': 500, 'HTD Lead Time': 90, 'HTD Rework': nan, 'HTD Completion': 80, 'HTD Readiness': nan, 'HTD Not Ready': nan, 'UAT Pass Rate': 100, 'UAT Completion': 80, 'UAT Rework': nan, 'UAT at Risk': nan, 'GLD Ticket Threshold P1': 0, 'GLD Ticket Threshold P2': 2, 'Go‑Live Success Ratio': 100, 'Hypercare Incident P1': 0, 'Hypercare Incident P2': 2, 'TTS Lead Time': 42, 'TTS Completion': 80, 'Billable Utilization': 80, 'NPS': 66, 'Extended Hypercare Cases': 42}


In [0]:
%sql
DESCRIBE TABLE qa_wb.saasfactory.g_dim_cpm_kpi_thresholds          

col_name,data_type,comment
high_risk_projects,int,null
sow,int,null
high_value_projects_dasb,int,null
htd_lead_time,int,null
htd_rework,int,null
htd_completion,int,null
htd_readiness,int,null
htd_not_ready,int,null
uat_pass_rate,int,null
uat_completion,int,null


In [0]:
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import col

# Target column names, in order
target_columns = [
    "high_risk_projects",
    "sow",
    "high_value_projects_dasb",
    "htd_lead_time",
    "htd_rework",
    "htd_completion",
    "htd_readiness",
    "htd_not_ready",
    "uat_pass_rate",
    "uat_completion",
    "uat_rework",
    "uat_at_risk",
    "gld_ticket_threshold_p1",
    "gld_ticket_threshold_p2",
    "go_live_success_ratio",
    "hypercare_incident_p1",
    "hypercare_incident_p2",
    "tts_lead_time",
    "tts_completion",
    "billable_utilization",
    "nps",
    "extended_hypercare_cases",
]

# Sanity check before blindly renaming by position
if len(df.columns) != len(target_columns):
    raise ValueError(
        f"Column count mismatch: df has {len(df.columns)} columns, "
        f"expected {len(target_columns)}. "
        f"df columns: {list(df.columns)}"
    )

# Rename by position
df.columns = target_columns

# Coerce to numeric, NaN-safe (still pandas-side cleanup)
for col_name in target_columns:
    df[col_name] = pd.to_numeric(df[col_name], errors="coerce").astype("Int64")

print(df.head())
print(df.dtypes)

# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Force every target column to 32-bit int to match the Delta table exactly
for col_name in target_columns:
    spark_df = spark_df.withColumn(col_name, col(col_name).cast(IntegerType()))

spark_df.printSchema()  # sanity check — confirm all columns now show as 'integer'

# Append to the existing table
spark_df.write.mode("append").saveAsTable("qa_wb.saasfactory.g_dim_cpm_kpi_thresholds")

   high_risk_projects  sow  ...  nps  extended_hypercare_cases
0                <NA>  100  ...   66                        42

[1 rows x 22 columns]
high_risk_projects          Int64
sow                         Int64
high_value_projects_dasb    Int64
htd_lead_time               Int64
htd_rework                  Int64
htd_completion              Int64
htd_readiness               Int64
htd_not_ready               Int64
uat_pass_rate               Int64
uat_completion              Int64
uat_rework                  Int64
uat_at_risk                 Int64
gld_ticket_threshold_p1     Int64
gld_ticket_threshold_p2     Int64
go_live_success_ratio       Int64
hypercare_incident_p1       Int64
hypercare_incident_p2       Int64
tts_lead_time               Int64
tts_completion              Int64
billable_utilization        Int64
nps                         Int64
extended_hypercare_cases    Int64
dtype: object
root
 |-- high_risk_projects: integer (nullable = true)
 |-- sow: integer (nullable = tru

In [0]:
%sql
select * from qa_wb.saasfactory.g_dim_cpm_kpi_thresholds

high_risk_projects,sow,high_value_projects_dasb,htd_lead_time,htd_rework,htd_completion,htd_readiness,htd_not_ready,uat_pass_rate,uat_completion,uat_rework,uat_at_risk,gld_ticket_threshold_p1,gld_ticket_threshold_p2,go_live_success_ratio,hypercare_incident_p1,hypercare_incident_p2,tts_lead_time,tts_completion,billable_utilization,nps,extended_hypercare_cases
null,100,500,90,null,80,null,null,100,80,null,null,0,2,100,0,2,42,80,80,66,42
